# Embeddings, Semantic Search & Retrieval

**Runtime:** Local LLMs & Embeddings via [Ollama](https://ollama.com)  
**Vector Store:** FAISS (Facebook AI Similarity Search)

---

### What you will learn
1. Generate & inspect embeddings
2. Compute similarity (cosine, euclidean)
3. Store embeddings in FAISS and search
4. Compare: Keyword vs BM25 vs Semantic vs Hybrid search
5. Metadata filtering
6. RAG — generate answers from retrieved context

### Core Idea
An **embedding** = a vector of numbers that captures **meaning**.  
Similar texts → close vectors. Different texts → far apart.  
This lets us search by **meaning**, not just keywords.

```
Documents → Chunk → Embed → Store in FAISS
Query → Embed → Search FAISS → Rank → Return Top-K → LLM Answer
```

### Our Scenario

We're building an internal search engine for **ZenithAI** — a fictional AI startup.  
Their wiki has product docs, team info, and policies that **no LLM has ever seen**.  
This makes retrieval *essential* — without it, the LLM has no clue about ZenithAI.

> **Prerequisite:** `ollama serve` running with `nomic-embed-text` and `llama3.2:3b` pulled.


In [ ]:
# ── Setup & Health Check ─────────────────────────────────────────────────────
import ollama
import numpy as np
import faiss
import math, time
from collections import Counter

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2:3b"

models = [m.model for m in ollama.list().models]
print("✅ Ollama is running")
print(f"   Models: {models}")
for m in [EMBED_MODEL, LLM_MODEL]:
    print(f"   {'✅' if any(m in x for x in models) else '⚠️  pull it →'} {m}")

✅ Ollama is running
   Models: ['llama3.1:latest', 'nomic-embed-text:latest', 'llama3.2:latest', 'quay.io/redhat-ai-services/modelcar-catalog:ibm-granite--granite-guardian-hap-38m', 'granite-code:3b', 'llama3.2:3b']
   ✅ nomic-embed-text
   ✅ llama3.2:3b


In [ ]:
# ── Helper Functions (reused throughout) ─────────────────────────────────────

def embed(text):
    """Get embedding vector for text."""
    return ollama.embed(model=EMBED_MODEL, input=text).embeddings[0]

def cosine_sim(a, b):
    """Cosine similarity: 1=identical, 0=unrelated."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def generate(prompt, temperature=0.3):
    """Generate text with LLM."""
    r = ollama.generate(model=LLM_MODEL, prompt=prompt,
                        options={"temperature": temperature, "num_predict": 300})
    return r.response

print("✅ Helpers ready: embed(), cosine_sim(), generate()")

✅ Helpers ready: embed(), cosine_sim(), generate()


---
## Step 1: Generate & Inspect Embeddings

Each text → a **768-dimensional vector**. Similar meanings → similar vectors.


In [ ]:
text = "ZenithAI's flagship product is called Prism"
vec = embed(text)

print(f"Text: \"{text}\"")
print(f"Dimensions: {len(vec)}")
print(f"First 10 values: {[f'{v:.4f}' for v in vec[:10]]}")
print(f"Stats: min={min(vec):.4f}, max={max(vec):.4f}, mean={np.mean(vec):.4f}")

Text: "ZenithAI's flagship product is called Prism"
Dimensions: 768
First 10 values: ['0.0102', '0.0638', '-0.1674', '0.0256', '0.0263', '-0.0496', '0.0176', '-0.0098', '0.0406', '0.0413']
Stats: min=-0.1674, max=0.1104, mean=0.0002


In [ ]:
# Sentences about our fictional company — notice the relationships!
sentences = [
    "Prism uses vector search to find relevant documents",
    "ZenithAI's product retrieves information using embeddings",  # same meaning!
    "The engineering team at ZenithAI has 42 members",            # same company, different topic
    "Jupiter is the largest planet in our solar system",          # totally unrelated
    "Orion is ZenithAI's internal deployment tool",               # same company, different product
    "ZenithAI uses Orion to ship code to production",            # same as above!
]

vecs = [embed(s) for s in sentences]

# Similarity matrix
print("COSINE SIMILARITY MATRIX (1.0=identical, 0=unrelated)\n")
print(f"{'':>55}", " ".join(f"[{i}]" for i in range(6)))
for i in range(6):
    row = " ".join(f"{cosine_sim(vecs[i], vecs[j]):.2f}" for j in range(6))
    print(f"[{i}] {sentences[i]:<52} {row}")

print("\n🔑 Key observations:")
print(f"  'Prism uses vector search' ↔ 'product retrieves using embeddings': {cosine_sim(vecs[0], vecs[1]):.4f} (HIGH!)")
print(f"  'Prism uses vector search' ↔ 'Jupiter is largest planet':          {cosine_sim(vecs[0], vecs[3]):.4f} (LOW)")
print(f"  'Orion is deployment tool' ↔ 'uses Orion to ship code':            {cosine_sim(vecs[4], vecs[5]):.4f} (HIGH!)")

COSINE SIMILARITY MATRIX (1.0=identical, 0=unrelated)

                                                        [0] [1] [2] [3] [4] [5]
[0] Prism uses vector search to find relevant documents  1.00 0.57 0.44 0.37 0.48 0.43
[1] ZenithAI's product retrieves information using embeddings 0.57 1.00 0.64 0.42 0.70 0.70
[2] The engineering team at ZenithAI has 42 members      0.44 0.64 1.00 0.47 0.66 0.62
[3] Jupiter is the largest planet in our solar system    0.37 0.42 0.47 1.00 0.49 0.46
[4] Orion is ZenithAI's internal deployment tool         0.48 0.70 0.66 0.49 1.00 0.80
[5] ZenithAI uses Orion to ship code to production       0.43 0.70 0.62 0.46 0.80 1.00

🔑 Key observations:
  'Prism uses vector search' ↔ 'product retrieves using embeddings': 0.5704 (HIGH!)
  'Prism uses vector search' ↔ 'Jupiter is largest planet':          0.3677 (LOW)
  'Orion is deployment tool' ↔ 'uses Orion to ship code':            0.8041 (HIGH!)


---
## Step 2: FAISS — Vector Database

FAISS stores vectors and finds nearest neighbors fast.  
`IndexFlatIP` with normalized vectors = cosine similarity search.


In [ ]:
dim = len(vecs[0])
index = faiss.IndexFlatIP(dim)

vectors = np.array(vecs, dtype=np.float32)
faiss.normalize_L2(vectors)
index.add(vectors)
print(f"✅ FAISS index: {index.ntotal} vectors × {dim} dims")

# Search with a query about our fictional company
query = "How does ZenithAI's search product work?"
q = np.array([embed(query)], dtype=np.float32)
faiss.normalize_L2(q)
scores, ids = index.search(q, k=3)

print(f"\n🔍 Query: \"{query}\"")
for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
    print(f"  #{rank} [{score:.4f}] \"{sentences[idx]}\"")
print("\n💡 Found Prism-related content — even though query says 'search product'!")

✅ FAISS index: 6 vectors × 768 dims

🔍 Query: "How does ZenithAI's search product work?"
  #1 [0.7762] "ZenithAI's product retrieves information using embeddings"
  #2 [0.6428] "Orion is ZenithAI's internal deployment tool"
  #3 [0.6106] "ZenithAI uses Orion to ship code to production"

💡 Found Prism-related content — even though query says 'search product'!


---
## Step 3: Load Knowledge Base & Build Search Index

This is the **ZenithAI internal wiki** — completely fictional information that no LLM knows.  
This makes retrieval genuinely necessary (the LLM can't answer without these chunks).


In [ ]:
documents = [
    ("Prism is ZenithAI's flagship product — an enterprise semantic search engine launched in March 2024. It processes 2.3 million queries per day across 47 enterprise clients. Prism uses a proprietary embedding model called Nova-7 with 1024 dimensions.",
     "Product Docs", "prism", "engineering"),
    ("Orion is ZenithAI's internal CI/CD and deployment platform. All production deployments must go through Orion. It requires two approvals for production pushes and automatically runs the Sentinel test suite before deploying.",
     "Engineering Wiki", "orion", "engineering"),
    ("The Pricing for Prism is tier-based: Starter plan at $499/month (up to 100k queries), Growth plan at $1,999/month (up to 1M queries), and Enterprise plan with custom pricing. All plans include a 14-day free trial.",
     "Sales Docs", "pricing", "sales"),
    ("ZenithAI was founded in January 2022 by Dr. Maya Chen (CEO) and Raj Nakamura (CTO). The company is headquartered in Austin, Texas with a remote-first policy. Currently employs 156 people across 12 countries.",
     "Company Wiki", "founding", "general"),
    ("Nova-7 is ZenithAI's proprietary embedding model trained on 840GB of curated text. It produces 1024-dimensional vectors and outperforms OpenAI's ada-002 by 12% on the MTEB benchmark. Training took 3 weeks on 64 A100 GPUs.",
     "Research Docs", "nova7", "engineering"),
    ("ZenithAI's Series B round closed in September 2023 at $67M valuation, led by Horizon Ventures with participation from TechFlow Capital. The round raised $18.5M to expand the sales team and launch Prism 2.0.",
     "Company Wiki", "funding", "general"),
    ("Sentinel is ZenithAI's automated testing framework. It runs 4,200 test cases before every deployment. Tests are categorized as: smoke (P0), integration (P1), regression (P2), and performance (P3). P0 failures block all deploys.",
     "Engineering Wiki", "sentinel", "engineering"),
    ("The Customer Success team is led by Sarah Park (VP of CS). They maintain a 97.3% retention rate across enterprise clients. Each client is assigned a dedicated Customer Success Manager (CSM) within 48 hours of signing.",
     "Team Wiki", "customer_success", "sales"),
    ("Prism's architecture uses a three-stage retrieval pipeline: Stage 1 (Recall) uses HNSW indices for fast candidate retrieval, Stage 2 (Rerank) applies a cross-encoder for precision, Stage 3 (Filter) applies metadata and permission filters.",
     "Product Docs", "architecture", "engineering"),
    ("ZenithAI's PTO policy: unlimited PTO with a minimum of 15 days per year enforced. Engineering on-call rotation is one week every 8 weeks. On-call engineers get a $500 bonus per rotation and next Monday off.",
     "HR Docs", "pto_policy", "hr"),
    ("Project Lighthouse is ZenithAI's codename for their upcoming multimodal search feature (Q1 2025 launch). It will support image, PDF, and audio search alongside text. Led by Principal Engineer Alex Okafor.",
     "Product Docs", "lighthouse", "engineering"),
    ("The sales team uses a MEDDIC qualification framework. Deals above $50K require VP approval. The average enterprise deal cycle is 47 days. Top performer last quarter: Marcus Rivera ($2.1M closed).",
     "Sales Docs", "sales_process", "sales"),
    ("ZenithAI's engineering team follows a 2-week sprint cycle. Code reviews require at least one approval from a senior engineer. All PRs must pass Sentinel checks and maintain >85% test coverage.",
     "Engineering Wiki", "dev_process", "engineering"),
    ("Prism's SLA guarantees: 99.95% uptime, p95 latency under 200ms, and p99 under 500ms. Breaches trigger automatic credits: 10% for uptime violations, 5% for latency violations. Monitored via Grafana dashboards.",
     "Product Docs", "sla", "engineering"),
    ("New employee onboarding at ZenithAI takes 2 weeks. Week 1: setup, culture, and buddy pairing. Week 2: team-specific training and first ticket assignment. All new hires present a 'Hello ZenithAI' talk on Day 10.",
     "HR Docs", "onboarding", "hr"),
]

# Build chunks with metadata
chunks = [{'id': i, 'text': t, 'metadata': {'source': s, 'topic': tp, 'dept': dept}}
          for i, (t, s, tp, dept) in enumerate(documents)]

# Embed and build FAISS index
print(f"📚 {len(chunks)} chunks — embedding...")
start = time.time()
chunk_vecs = [embed(c['text']) for c in chunks]
print(f"   Done in {time.time()-start:.1f}s\n")

dim = len(chunk_vecs[0])
chunk_matrix = np.array(chunk_vecs, dtype=np.float32)
faiss.normalize_L2(chunk_matrix)
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_matrix)

print(f"✅ FAISS index: {faiss_index.ntotal} vectors × {dim} dims")
for c in chunks:
    print(f"  [{c['id']:2d}] {c['metadata']['topic']:<18} ({c['metadata']['source']}, {c['metadata']['dept']})")

📚 15 chunks — embedding...


   Done in 0.6s

✅ FAISS index: 15 vectors × 768 dims
  [ 0] prism              (Product Docs, engineering)
  [ 1] orion              (Engineering Wiki, engineering)
  [ 2] pricing            (Sales Docs, sales)
  [ 3] founding           (Company Wiki, general)
  [ 4] nova7              (Research Docs, engineering)
  [ 5] funding            (Company Wiki, general)
  [ 6] sentinel           (Engineering Wiki, engineering)
  [ 7] customer_success   (Team Wiki, sales)
  [ 8] architecture       (Product Docs, engineering)
  [ 9] pto_policy         (HR Docs, hr)
  [10] lighthouse         (Product Docs, engineering)
  [11] sales_process      (Sales Docs, sales)
  [12] dev_process        (Engineering Wiki, engineering)
  [13] sla                (Product Docs, engineering)
  [14] onboarding         (HR Docs, hr)


---
## Step 4: Four Retrieval Methods

| Method | How | Strengths | Weaknesses |
|--------|-----|-----------|------------|
| Keyword | Word overlap | Fast | Misses synonyms |
| BM25 | TF-IDF + length norm | Better weighting | Still keyword-bound |
| **Semantic** ⭐ | Embedding similarity | Understands meaning | Heavier compute |
| Hybrid | BM25 + Semantic | Best of both | Needs tuning |


In [ ]:
# ── Keyword Search ────────────────────────────────────────────────────────────
def keyword_search(query, top_k=5):
    q_words = set(query.lower().split())
    scored = [(c, len(q_words & set(c['text'].lower().split())) / len(q_words)) for c in chunks]
    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

# ── BM25 ──────────────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b, self.n = k1, b, len(docs)
        self.tokens = [d.lower().split() for d in docs]
        self.avg_dl = sum(len(t) for t in self.tokens) / self.n
        self.df = {}
        for toks in self.tokens:
            for t in set(toks): self.df[t] = self.df.get(t, 0) + 1

    def search(self, query, top_k=5):
        scores = []
        for i, toks in enumerate(self.tokens):
            tf = Counter(toks)
            s = 0
            for term in query.lower().split():
                if term not in tf: continue
                idf = math.log((self.n - self.df.get(term,0) + 0.5) / (self.df.get(term,0) + 0.5) + 1)
                s += idf * tf[term] * (self.k1+1) / (tf[term] + self.k1*(1-self.b + self.b*len(toks)/self.avg_dl))
            scores.append((i, s))
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

bm25 = BM25([c['text'] for c in chunks])

# ── Semantic Search ───────────────────────────────────────────────────────────
def semantic_search(query, top_k=5, filter_fn=None):
    q = np.array([embed(query)], dtype=np.float32)
    faiss.normalize_L2(q)
    scores, ids = faiss_index.search(q, len(chunks))
    results = []
    for idx, sc in zip(ids[0], scores[0]):
        if filter_fn and not filter_fn(chunks[idx]): continue
        results.append((chunks[idx], float(sc)))
        if len(results) >= top_k: break
    return results

# ── Hybrid Search ─────────────────────────────────────────────────────────────
def hybrid_search(query, top_k=5, alpha=0.5):
    bm_scores = {i: s for i, s in bm25.search(query, len(chunks))}
    bm_max = max(bm_scores.values()) or 1
    q = np.array([embed(query)], dtype=np.float32)
    faiss.normalize_L2(q)
    sem_sc, sem_ids = faiss_index.search(q, len(chunks))
    sem_map = {int(i): float(s) for i, s in zip(sem_ids[0], sem_sc[0])}
    combined = [(chunks[i], alpha*sem_map.get(i,0) + (1-alpha)*bm_scores.get(i,0)/bm_max) for i in range(len(chunks))]
    return sorted(combined, key=lambda x: x[1], reverse=True)[:top_k]

print("✅ All methods ready: keyword_search, bm25, semantic_search, hybrid_search")

✅ All methods ready: keyword_search, bm25, semantic_search, hybrid_search


---
## Step 5: Head-to-Head Comparison


In [ ]:
queries = [
    "How much does Prism cost?",
    "Who founded the company and when?",
    "What happens before code goes to production?",
    "How does the product find relevant results so fast?",
]

print("🏆 HEAD-TO-HEAD COMPARISON\n")
for query in queries:
    kw = keyword_search(query, 1)
    bm = bm25.search(query, 1)
    sem = semantic_search(query, 1)
    hyb = hybrid_search(query, 1)
    print(f"🔍 \"{query}\"")
    print(f"   Keyword:  [{kw[0][1]:.2f}] {kw[0][0]['metadata']['topic']}")
    print(f"   BM25:     [{bm[0][1]:.2f}] {chunks[bm[0][0]]['metadata']['topic']}")
    print(f"   Semantic: [{sem[0][1]:.4f}] {sem[0][0]['metadata']['topic']}")
    print(f"   Hybrid:   [{hyb[0][1]:.4f}] {hyb[0][0]['metadata']['topic']}")
    print()

🏆 HEAD-TO-HEAD COMPARISON

🔍 "How much does Prism cost?"
   Keyword:  [0.20] prism
   BM25:     [2.09] prism
   Semantic: [0.8092] pricing
   Hybrid:   [0.8469] prism

🔍 "Who founded the company and when?"
   Keyword:  [0.67] founding
   BM25:     [5.72] founding
   Semantic: [0.6396] founding
   Hybrid:   [0.8198] founding



🔍 "What happens before code goes to production?"
   Keyword:  [0.14] orion
   BM25:     [2.60] pricing
   Semantic: [0.5274] dev_process
   Hybrid:   [0.7378] dev_process

🔍 "How does the product find relevant results so fast?"
   Keyword:  [0.11] prism
   BM25:     [2.24] prism
   Semantic: [0.5461] customer_success
   Hybrid:   [0.7728] prism



---
## ⭐ Step 6: Why Semantic Search Wins

These queries describe what the user *wants to know* without using terms from the wiki.  
Only semantic search understands the intent.


In [ ]:
tests = [
    ("How do I ship my code to users?", "orion"),
    ("What's the vacation policy here?", "pto_policy"),
    ("Tell me about the AI model ZenithAI trained", "nova7"),
    ("What's coming next for the product?", "lighthouse"),
    ("How do new hires get started?", "onboarding"),
]

print("⭐ SEMANTIC vs BM25 — Paraphrased Queries\n")
sem_ok = bm_ok = 0
for query, expected in tests:
    s_topic = semantic_search(query, 1)[0][0]['metadata']['topic']
    b_topic = chunks[bm25.search(query, 1)[0][0]]['metadata']['topic']
    sem_ok += (s_topic == expected)
    bm_ok += (b_topic == expected)
    print(f"  \"{query}\"")
    print(f"    Expected: {expected}")
    print(f"    Semantic → {s_topic} {'✅' if s_topic==expected else '❌'}")
    print(f"    BM25     → {b_topic} {'✅' if b_topic==expected else '❌'}\n")

print(f"Score: Semantic {sem_ok}/{len(tests)} | BM25 {bm_ok}/{len(tests)}")
print("\n🔑 Semantic search finds 'orion' when you ask 'how to ship code' — no word overlap needed!")

⭐ SEMANTIC vs BM25 — Paraphrased Queries

  "How do I ship my code to users?"
    Expected: orion
    Semantic → onboarding ❌
    BM25     → pricing ❌

  "What's the vacation policy here?"
    Expected: pto_policy
    Semantic → pto_policy ✅
    BM25     → sales_process ❌

  "Tell me about the AI model ZenithAI trained"
    Expected: nova7
    Semantic → nova7 ✅
    BM25     → nova7 ✅



  "What's coming next for the product?"
    Expected: lighthouse
    Semantic → funding ❌
    BM25     → pto_policy ❌

  "How do new hires get started?"
    Expected: onboarding
    Semantic → onboarding ✅
    BM25     → onboarding ✅

Score: Semantic 3/5 | BM25 2/5

🔑 Semantic search finds 'orion' when you ask 'how to ship code' — no word overlap needed!


---
## Step 7: Scoring, Ranking & Metadata Filtering


In [ ]:
query = "Tell me about ZenithAI's engineering practices"
print(f"🔍 \"{query}\"\n")

# Full ranking
all_results = semantic_search(query, top_k=len(chunks))
print("Full ranking (cosine similarity):")
for rank, (chunk, score) in enumerate(all_results, 1):
    bar = '█' * int(score * 20)
    print(f"  {'⭐' if score > 0.6 else '  '}#{rank:<2} {score:.4f} {bar:<12} {chunk['metadata']['topic']}")

# Metadata filtering
print(f"\n🏷️  Filter: engineering dept only:")
for chunk, score in semantic_search(query, 3, filter_fn=lambda c: c['metadata']['dept']=='engineering'):
    print(f"  [{score:.4f}] {chunk['metadata']['topic']} ({chunk['metadata']['source']})")

print(f"\n🏷️  Filter: HR docs only:")
for chunk, score in semantic_search(query, 3, filter_fn=lambda c: c['metadata']['dept']=='hr'):
    print(f"  [{score:.4f}] {chunk['metadata']['topic']} ({chunk['metadata']['source']})")

🔍 "Tell me about ZenithAI's engineering practices"

Full ranking (cosine similarity):
  ⭐#1  0.7930 ███████████████ dev_process
  ⭐#2  0.7475 ██████████████ founding
  ⭐#3  0.7071 ██████████████ pto_policy
  ⭐#4  0.6624 █████████████ funding
  ⭐#5  0.6557 █████████████ orion
  ⭐#6  0.6451 ████████████ sentinel
  ⭐#7  0.6394 ████████████ onboarding
  ⭐#8  0.6305 ████████████ prism
  ⭐#9  0.6175 ████████████ lighthouse
    #10 0.5879 ███████████  nova7
    #11 0.5080 ██████████   architecture
    #12 0.4651 █████████    sla
    #13 0.4564 █████████    customer_success
    #14 0.4464 ████████     pricing
    #15 0.4276 ████████     sales_process

🏷️  Filter: engineering dept only:
  [0.7930] dev_process (Engineering Wiki)
  [0.6557] orion (Engineering Wiki)
  [0.6451] sentinel (Engineering Wiki)

🏷️  Filter: HR docs only:
  [0.7071] pto_policy (HR Docs)
  [0.6394] onboarding (HR Docs)


---
## Step 8: RAG — Generate Answers from Retrieved Context

This is where retrieval **truly shines**. The LLM knows nothing about ZenithAI.  
Without retrieval → it hallucinates. With retrieval → accurate, grounded answers.


In [ ]:
def rag(query, top_k=3):
    """Retrieve context + generate answer."""
    retrieved = semantic_search(query, top_k)
    context = "\n\n".join(f"[Source {i+1}: {c['metadata']['source']}]\n{c['text']}"
                           for i, (c, _) in enumerate(retrieved))
    prompt = f"""Answer based ONLY on context below. Cite [Source N]. Be concise.

CONTEXT:\n{context}\n\nQUESTION: {query}\nANSWER:"""
    return generate(prompt), retrieved

# Questions only answerable with our fictional wiki
for q in ["How much does the Growth plan cost and what's included?",
          "Who leads the Customer Success team and what's the retention rate?",
          "What is Project Lighthouse and when does it launch?"]:
    print(f"{'━'*70}")
    print(f"❓ {q}")
    answer, retrieved = rag(q)
    print(f"  📥 Retrieved: {[c['metadata']['topic'] for c, _ in retrieved]}")
    print(f"  💬 {answer[:300]}\n")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❓ How much does the Growth plan cost and what's included?


  📥 Retrieved: ['pricing', 'funding', 'pto_policy']
  💬 The Growth plan costs $1,999/month and includes up to 1M queries. [Source N: Source 1 - Sales Docs]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❓ Who leads the Customer Success team and what's the retention rate?


  📥 Retrieved: ['customer_success', 'onboarding', 'sales_process']
  💬 Sarah Park (VP of CS) leads the Customer Success team, with a 97.3% retention rate across enterprise clients. [Source N: Team Wiki]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❓ What is Project Lighthouse and when does it launch?


  📥 Retrieved: ['lighthouse', 'orion', 'sentinel']
  💬 Project Lighthouse is ZenithAI's upcoming multimodal search feature, supporting image, PDF, audio, and text search. It launches in Q1 2025. [Source N: Product Docs]



In [ ]:
# THE KEY DEMO: RAG vs No-RAG on fictional info
q = "Who founded ZenithAI, when, and how much funding have they raised?"
print(f"🔍 \"{q}\"\n")

print("🟢 WITH RAG (has access to ZenithAI wiki):")
print(rag(q)[0][:400])

print("\n🔴 WITHOUT RAG (LLM has NEVER seen this info):")
print(generate(f"Answer this question concisely: {q}")[:400])

print("\n💡 The LLM either refuses or HALLUCINATES without retrieval.")
print("   With RAG, it gives the correct answer: Dr. Maya Chen & Raj Nakamura, Jan 2022, $18.5M Series B.")

🔍 "Who founded ZenithAI, when, and how much funding have they raised?"

🟢 WITH RAG (has access to ZenithAI wiki):


Dr. Maya Chen (CEO) and Raj Nakamura (CTO) founded ZenithAI in January 2022. The company has raised $67M in Series B funding, with $18.5M of that coming from the round closed in September 2023. [Source N: Company Wiki]

🔴 WITHOUT RAG (LLM has NEVER seen this info):


I couldn't find any information on a company called "ZenithAI". It's possible that it's a private or relatively new company. Can you provide more context or details about ZenithAI? I'll do my best to help.

💡 The LLM either refuses or HALLUCINATES without retrieval.
   With RAG, it gives the correct answer: Dr. Maya Chen & Raj Nakamura, Jan 2022, $18.5M Series B.


---
## 🎮 Try It Yourself


In [ ]:
my_query = "What tests run before deployment and what blocks a release?"  # ← change this!

print(f"🔍 \"{my_query}\"\n")
print("Semantic:"); [print(f"  [{s:.4f}] {c['metadata']['topic']}") for c, s in semantic_search(my_query, 3)]
print("\nBM25:"); [print(f"  [{s:.4f}] {chunks[i]['metadata']['topic']}") for i, s in bm25.search(my_query, 3)]
print("\n💬 RAG Answer:")
print(rag(my_query)[0][:400])

🔍 "What tests run before deployment and what blocks a release?"

Semantic:
  [0.6522] sentinel
  [0.6132] orion
  [0.5376] dev_process

BM25:
  [4.6893] orion
  [4.5199] sentinel
  [1.0224] dev_process

💬 RAG Answer:


According to the Engineering Wiki, Sentinel runs all 4,200 test cases before every deployment, with P0 failures blocking all deploys. [Source N: Source 1]


---
## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Embeddings | 768-dim vectors capturing semantic meaning |
| Cosine Similarity | Best metric for comparing text embeddings |
| FAISS | Fast vector search (IndexFlatIP + normalization = cosine) |
| Keyword/BM25 | Good for exact terms, fails on synonyms |
| **Semantic Search** ⭐ | Finds results by meaning — even with zero word overlap |
| Hybrid | BM25 + Semantic combined — production choice |
| RAG | Essential when LLM doesn't know the info — retrieve then generate |

### Why this demo works
The LLM has **never seen** ZenithAI's wiki during training.  
Without retrieval → it hallucinates or says "I don't know".  
With retrieval → accurate answers grounded in our documents.  
**That's the whole point of RAG.**
